# Experiment 1 — XGBoost Baseline (No Imbalance Handling)

Standard XGBoost with zero modification. Worst-case reference point.
Shows how badly standard training fails on imbalanced data.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# FIXED HYPERPARAMETERS — DO NOT CHANGE ACROSS ANY XGB EXPERIMENT
# FIXED: removed use_label_encoder=False (removed in XGBoost 2.0+, causes TypeError)
# ADDED: tree_method='hist' — much faster on large datasets (11M rows)
# ADDED: n_jobs=-1 — use all CPU cores
# ADDED: device='cpu' — explicit, avoids XGBoost 2.x warnings
PARAMS = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='auc',
    verbosity=0,
    tree_method='hist',
    device='cpu',
    n_jobs=-1
)

print("Experiment 1 — XGBoost Baseline")
print("Params:", PARAMS)

In [ ]:
all_results = []

for v in ['A', 'B', 'C']:
    print(f"\n[Exp1-XGB] Training Version {v}...")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label', axis=1).values
    y_train = train['label'].values
    X_test  = test.drop('label', axis=1).values
    y_test  = test['label'].values

    model = XGBClassifier(**PARAMS)
    t0    = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    probs = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)

    metrics = compute_all_metrics(y_test, probs, preds, train_time)
    metrics['Version'] = v
    all_results.append(metrics)

    np.save(os.path.join(RESULTS_DIR, f'probs_exp1_xgb_version_{v}.npy'), probs)
    np.save(os.path.join(RESULTS_DIR, f'labels_version_{v}.npy'), y_test)
    print_metrics(metrics, label=f"Exp1-XGB Version {v}")

    # Free memory after each version — important for 11M row full runs
    del train, test, X_train, y_train, X_test, y_test, model, probs, preds

print("\n[Exp1-XGB] Complete.")

In [ ]:
results_df = pd.DataFrame(all_results)[['Version', 'AUC', 'F1', 'Signal_Significance', 'Train_Time_sec']]
save_path  = os.path.join(RESULTS_DIR, 'experiment1_baseline_xgb.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")